In [68]:
import pandas as pd
import torch 
import matplotlib as pyplt
import tqdm 

In [69]:
df = pd.read_csv('/kaggle/input/datasets/organizations/nih-chest-xrays/data/Data_Entry_2017.csv')
df.head()

,Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],Unnamed: 11
0,00000001_000.png,Cardiomegaly,0,1,58,M,PA,2682,2749,0.143,0.143,NaN
1,00000001_001.png,Cardiomegaly|Emphysema,1,1,58,M,PA,2894,2729,0.143,0.143,NaN
2,00000001_002.png,Cardiomegaly|Effusion,2,1,58,M,PA,2500,2048,0.168,0.168,NaN
3,00000002_000.png,No Finding,0,2,81,M,PA,2500,2048,0.171,0.171,NaN
4,00000003_000.png,Hernia,0,3,81,F,PA,2582,2991,0.143,0.143,NaN


In [70]:
#parsing the pipe string into list for training 
from sklearn.preprocessing import MultiLabelBinarizer
df['labels'] = df['Finding Labels'].apply(lambda x: x.split('|')) 

mlb = MultiLabelBinarizer()
labels = mlb.fit_transform(df['labels'])

disease_cols = list(mlb.classes_)

# Drop these columns if they already exist before concat
df = df.drop(columns=[c for c in disease_cols if c in df.columns])
df = df.reset_index(drop=True)

label_df = pd.DataFrame(labels, columns=disease_cols)
df = pd.concat([df, label_df], axis=1)

print(df.columns.tolist())
print(df[disease_cols].head())

['Image Index', 'Finding Labels', 'Follow-up #', 'Patient ID', 'Patient Age', 'Patient Gender', 'View Position', 'OriginalImage[Width', 'Height]', 'OriginalImagePixelSpacing[x', 'y]', 'Unnamed: 11', 'labels', 'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Effusion', 'Emphysema', 'Fibrosis', 'Hernia', 'Infiltration', 'Mass', 'No Finding', 'Nodule', 'Pleural_Thickening', 'Pneumonia', 'Pneumothorax']
   Atelectasis  Cardiomegaly  Consolidation  Edema  Effusion  Emphysema  \
0            0             1              0      0         0          0   
1            0             1              0      0         0          1   
2            0             1              0      0         1          0   
3            0             0              0      0         0          0   
4            0             0              0      0         0          0   

   Fibrosis  Hernia  Infiltration  Mass  No Finding  Nodule  \
0         0       0             0     0           0       0   
1         0

In [71]:
from sklearn.model_selection import train_test_split
with open('/kaggle/input/datasets/organizations/nih-chest-xrays/data/train_val_list.txt') as f:
    train_val_files = set(f.read().splitlines())
with open('/kaggle/input/datasets/organizations/nih-chest-xrays/data/test_list.txt') as f:
    test_files = set(f.read().splitlines())

train_val_df = df[df['Image Index'].isin(train_val_files)].reset_index(drop=True)
test_df = df[df['Image Index'] .isin(test_files)].reset_index(drop=True)

#spliting pateints by ID and not randomly to prevent data leak 
patients = df['Patient ID'].unique()
train_patients, val_patients = train_test_split(patients, test_size=0.2, random_state=42) 

train_df = train_val_df[train_val_df['Patient ID'].isin(train_patients)].reset_index(drop=True)
val_df = train_val_df[train_val_df['Patient ID'].isin(val_patients)].reset_index(drop=True)

print(f" Train lenght:{len(train_df)} | test lenght:{len(test_df)} | val lenght:{len(val_df)}")

 Train lenght:69236 | test lenght:25596 | val lenght:17288


In [72]:
import os

base_dir = '/kaggle/input/datasets/organizations/nih-chest-xrays/data/'

image_paths = {}
for folder in os.listdir(base_dir):
    folder_path = os.path.join(base_dir, folder)
    if os.path.isdir(folder_path) and 'image' in folder.lower():
        images_subdir = os.path.join(folder_path, 'images')
        if os.path.isdir(images_subdir):
            for fname in os.listdir(images_subdir):
                image_paths[fname] = os.path.join(images_subdir, fname)

print(f'Total images found: {len(image_paths)}')

Total images found: 112120


In [73]:
#Creating a custom dataset class 
import os
import torch
from PIL import Image 
import numpy as np
from torch.utils.data import Dataset, DataLoader

class ChestXRay(Dataset):
    def __init__(self, df, image_paths, transform=None):
        self.df = df.reset_index(drop=True)
        self.image_paths = image_paths
        self.transform = transform
        self.df = df[disease_cols].values.astype(np.float32)
        
    def __len__(self):
        return len(self.df)
        
    def __getitem__(self, idx):
       img_name = self.df.loc[idx, ['Image Index']]
       img_paths = self.image_paths[img_name]
       image  = Image.open(image_path).convert('RGB')
       if self.transform:
           self.transform(image_path)
       labels = torch.tensor[self.labels[idx]]
       return images , labels

In [76]:
# train_transfrom and test_transform 
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
     transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])


In [77]:
#creating dataset 
train_dataset = ChestXRay(train_df, image_paths, transform=train_transform)
test_dataset  = ChestXRay(test_df,  image_paths, transform=val_test_transform)
val_dataset   = ChestXRay(val_df,   image_paths, transform=val_test_transform)

print(len(train_dataset), len(test_dataset), len(val_dataset))

69236 25596 17288


In [81]:
#Turning dataset into Dataloader
from torch.utils.data import DataLoader

trainloader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, drop_last=True)
testloader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2, drop_last=True)
valloader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2, drop_last=True)

print(len(trainloader), len(testloader), len(valloader))

2163 799 540
